In [ ]:
import os
import glob
import pandas as pd
from datetime import timedelta

# 경로 설정
target_dir = "/mnt/AEGIS/AEGIS/Deidentification/AEGIS-Order-CIV2/de"

# 환자별 노이즈 정보
noise_df = pd.read_csv("/mnt/AEGIS/AEGIS/Deidentification/AEGIS-PT/pt_noise_date_time.csv", sep=",", quotechar ='"', lineterminator = '\n',dtype = {'pt_id': str})

# 노이즈 매핑 딕셔너리
noise_df['new_pt_id'] = noise_df['new_pt_id'].astype(str)

pt_id_map = noise_df.set_index('pt_id')['new_pt_id'].to_dict()
date_noise_map = noise_df.set_index('new_pt_id')['date_noise'].astype(int).to_dict()
time_noise_map = noise_df.set_index('new_pt_id')['time_noise'].astype(int).to_dict()

# 대상 파일 수집
csv_files = glob.glob(os.path.join(target_dir, '**/*.csv'), recursive = True)
print(f"총 처리 대상 파일 수 : {len(csv_files)}")



In [ ]:
# ① pt_id 치환
print("\n① pt_id 치환 시작")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype={'pt_id':str},sep=',', quotechar='"', lineterminator='\n')
        
        if 'pt_id' in df.columns:
            df['pt_id'] = df['pt_id'].map(pt_id_map).fillna(df['pt_id'])
            
        print(df.head(3))
        
        df.to_csv(file, index=False, encoding = 'utf-8')
        print(f"✅ pt_id 저장 완료: {file}")
    except Exception as e:
        print(f"[❗ pt_id 오류] {file} → {e}")

In [ ]:
# ② hos_dt ±7일 ±30분
print("\n② hos_dt 노이즈 적용")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        
        if 'hos_dt' in df.columns:
            df['hos_dt'] = pd.to_datetime(df['hos_dt'], errors='coerce')
            df['hos_dt'] = df['hos_dt'] + df['pt_id'].map(lambda x: timedelta(days=date_noise_map.get(x, 0), minutes=time_noise_map.get(x, 0)))
            
            print(df.head(3))
            
            df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"✅ hos_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[❗ hos_dt 오류] {file} → {e}")



In [ ]:
# order_date 처리

for file in csv_files:

    try:

        df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')
        

        if 'order_date' in df.columns:

            df['order_date'] = pd.to_datetime(df['order_date'], errors = 'coerce')

            

            df['order_date'] = df['order_date'] \
                + pd.to_timedelta(df['pt_id'].map(date_noise_map).fillna(0).astype(int), unit = "D")
            

            print(df.head(3))

            

            df.to_csv(file, index = False, encoding = 'utf-8')

            print(f"order_date 완료: {file}")

            

    

    except Exception as e:

        print(f"[order_date 오류] {file} -> {e}")

In [ ]:
#  civ_dt ±7일 ±30분
print("\n civ_dt 노이즈 적용")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        
        if 'civ_dt' in df.columns:
            df['civ_dt'] = pd.to_datetime(df['civ_dt'], errors='coerce')
            df['civ_dt'] = df['civ_dt'] + df['pt_id'].map(lambda x: timedelta(days=date_noise_map.get(x, 0), minutes=time_noise_map.get(x, 0)))
            
            print(df.head(3))
            
            df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"✅ civ_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[❗ civ_dt 오류] {file} → {e}")



In [ ]:
# civ_med_weight_value 처리
import numpy as np

for file in csv_files:
    try:
        df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar='"', lineterminator='\n')

        if 'civ_med_weight_value' in df.columns:
            df['civ_med_weight_value'] = pd.to_numeric(df['civ_med_weight_value'], errors='coerce')

            # 2. 상하위 1% 경계값 계산
            lower_bound = df['civ_med_weight_value'].quantile(0.01)
            upper_bound = df['civ_med_weight_value'].quantile(0.99)

            # 3. 상하위 1% 바깥값은 경계값으로 대체 (clip)
            df['civ_med_weight_value'] = df['civ_med_weight_value'].clip(lower=lower_bound, upper=upper_bound)

            # 소수점 제거
            df['civ_med_weight_value'] = df['civ_med_weight_value'].apply(
            lambda x: np.floor(x) if pd.notna(x) else np.nan
            )
            
            print(df['civ_med_weight_value'])
            # 저장
            df.to_csv(file, index=False, encoding='utf-8')
            print(f"✅ civ_med_weight_value 처리 완료: {file}")

    except Exception as e:
        print(f"[❗ civ_med_weight_value 오류] {file} → {e}")

In [ ]:
#  order_nursing_dt ±7일 ±30분
print("\n order_nursing_dt 노이즈 적용")
for file in csv_files:
    try:
        df = pd.read_csv(file, dtype = {'pt_id':str}, sep=',', quotechar='"', lineterminator='\n')
        
        if 'order_nursing_dt' in df.columns:
            df['order_nursing_dt'] = pd.to_datetime(df['order_nursing_dt'], errors='coerce')
            df['order_nursing_dt'] = df['order_nursing_dt'] + df['pt_id'].map(lambda x: timedelta(days=date_noise_map.get(x, 0), minutes=time_noise_map.get(x, 0)))
            
            print(df.head(3))
            
            df.to_csv(file, index=False, encoding = 'utf-8')
            print(f"✅ order_nursing_dt 저장 완료: {file}")
    except Exception as e:
        print(f"[❗ order_nursing_dt 오류] {file} → {e}")



In [ ]:
# 250924 추가
#  medicine_order_date ±7일

for file in csv_files:

    try:

        df = pd.read_csv(file, dtype={'pt_id': str}, sep=",", quotechar ='"', lineterminator = '\n')
        

        if 'medicine_order_date' in df.columns:

            df['medicine_order_date'] = pd.to_datetime(df['medicine_order_date'], errors = 'coerce')

            

            df['medicine_order_date'] = df['medicine_order_date'] \
                + pd.to_timedelta(df['pt_id'].map(date_noise_map).fillna(0).astype(int), unit = "D")
            

            print(df.head(3))

            

            df.to_csv(file, index = False, encoding = 'utf-8-sig')

            print(f"order_date 완료: {file}")

            

    

    except Exception as e:

        print(f"[order_date 오류] {file} -> {e}")